Lab 1: Document Processing with OCR

In this lesson, you will build an agent for document processing with optical character recognition (OCR).

Learning Objectives:

    Parsing and extracting information from documents
    Building an agent equipped with an OCR tool
    Identifying failure modes of OCR

Background

Document processing converts unstructured documents meant for humans into structured data meant for machines. When documents are scanned images, OCR converts pixels to text. However, OCR alone produces raw text. Using LLMs, we can make sense of it. 

### 1- Make imports

In [ ]:
from PIL import Image

import pytesseract

from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor

from langchain_openai import ChatOpenAI

### 2- Create OCR tool

In [ ]:
from langchain.tools import tool

@tool
def ocr_read_doc(image_path: str) -> str:
    """Reads text from an image document using OCR."""
    try:
        text = pytesseract.image_to_string(Image.open(image_path))
        return text
    except Exception as e:
        return f"Error reading document: {e}"

#### 2.1- Use OCR to parse a sample document

In [ ]:
doc_path = 'files/TD_CASH_BACK_VISA_INFINITE__CARD_3226_Feb_06-2026.pdf'

In [ ]:
# turn PDF pages into images
import fitz
from PIL import Image

doc = fitz.open(doc_path)
for i, page in enumerate(doc):
    pix = page.get_pixmap()
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    img.save(f"files/page_{i}.png")

In [ ]:
from IPython.display import display

image_path = 'files/page_0.png'
img = Image.open(image_path)
display(img)

In [ ]:
import pytesseract

# pytesseract.pytesseract.tesseract_cmd = r"C:/Program Files/Tesseract-OCR/tesseract.exe"

In [ ]:
ocr_text = ocr_read_doc.run("files/invoice_sample.png")

print("raw OCR output:\n ---------------------\n", ocr_text)


#### 2.2. Use Regex to Extract Information

In [ ]:
import re
tax_match = re.search(r'Sales Tax\s*\$?([0-9.,]+)', ocr_text)
total_match = re.search(r'Total\s*\$?([0-9.,]+)', ocr_text)

if tax_match:
    print("tax: ", tax_match.group(1))
else:
    print("no match was found")

if total_match:
    print("total: ", total_match.group(1))
else:
    print("no match was found")

In [ ]:
import re

pattern = re.compile(
    r"(?i)\bSales\s*Tax\b\s*[:\-]?\s*([$€]?\s*\d+[.,]\d{2})"
)

match = pattern.search(ocr_text)
tax_value = match.group(1) if match else None
tax_value

In [ ]:
ocr_text

### 3- Create the agent

In [ ]:
import os
from dotenv import load_dotenv

_ = load_dotenv(override=True)

In [ ]:
# define the tools

tools = [ocr_read_doc]

# set up the OpenAI model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-mini", temperature=1)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent
from langchain_classic.agents import AgentExecutor

prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        "You are a helpful assistant for extracting information from documents."
        "You have access to the following tool for reading documents: ocr_read_doc(image_path: str) -> str. "
    ),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),

])

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

### 4. Run Agent and Extract Info

In [ ]:
task = """
Please process the document at 'files/invoice_sample.png' using the OCR tool 
and extract the following information in JSON format:
- tax
- total
"""

# Use .invoke() with a dictionary input for the agent_executor
response = agent_executor.invoke({"input": task})